In [12]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
#import clean_Dataset
data_path = "/content/drive/MyDrive/Colab Notebooks/Bitcoin_Project/Clean_Dataset.csv"
df = pd.read_csv(data_path)
df.head()

,timestamp,close,volume,number_of_trades,buy_pressure,log_volume,hour_sin,hour_cos,day_sin,day_cos,returns_scaled
0,2018-01-01 01:00:00,13203.06,383.697006,4534.0,0.471310,15.455385,0.258819,0.965926,0.0,1.0,-4.869355
1,2018-01-01 02:00:00,13330.18,429.064572,4887.0,0.448040,15.557763,0.500000,0.866025,0.0,1.0,1.929311
2,2018-01-01 03:00:00,13410.03,420.087030,4789.0,0.328309,15.548484,0.707107,0.707107,0.0,1.0,1.195848
3,2018-01-01 04:00:00,13601.01,340.807329,4563.0,0.507494,15.338965,0.866025,0.500000,0.0,1.0,2.859471
4,2018-01-01 05:00:00,13558.99,404.229046,5086.0,0.352105,15.520087,0.965926,0.258819,0.0,1.0,-0.634760


#EMA (Trend Detection)

he Exponential Moving Average (EMA) is a technical indicator used to identify the direction of the price trend. Unlike a Simple Moving Average (SMA), the EMA reacts faster to recent price changes because it gives more weight to the most recent data points.

In this project, we use two EMAs:

    EMA_12 (Short-term): Reflects the price trend over the last 12 hours.

    EMA_26 (Long-term): Reflects the price trend over the last 26 hours.

The Logic for the Model:

    Uptrend: If the close price is above the EMA, the market is generally moving up.

    Cross-over: If the Short-term EMA (12) crosses above the Long-term EMA (26), it is often a signal of strong "Buy" momentum.

    Distance Feature: We also calculate the "Distance" between the price and the EMA. This helps the AI understand if the price is getting too far away from the average (becoming "over-extended").

In [14]:
#create 12 hours EMA
df['ema_12'] = df['close'].ewm(span=12,adjust=False).mean()
df['ema_26'] = df['close'].ewm(span=26,adjust=False).mean() #Gerald Appel created an indicator called the MACD
df['ema_gap'] = (df['close']-df['ema_12'])/df['ema_12'] # 12 not 26 this represent market mood
#EMA Wearm up, so some the first rows will be drop firs 12 or 26 rows . as they will have same values of close column
df = df

#RSI (Relative Strength Index)

The Relative Strength Index (RSI) is a momentum oscillator that measures the speed and change of price movements. It oscillates between 0 and 100.

The "Overbought" vs. "Oversold" Logic:

    RSI > 70 (Overbought): The price has been rising very fast. The market might be "exhausted," and a downward correction or reversal is likely.

    RSI < 30 (Oversold): The price has been falling very fast. The market might be "undervalued," and a bounce upward is likely.

    RSI ~ 50 (Neutral): The market is in a stable trend with no extreme momentum.

In [15]:
def calculate_rsi(series,period=14):
  delta = series.diff()
  gain = (delta.where(delta>0,0)).rolling(window=period).mean()
  loss = (-delta.where(delta<0,0)).rolling(window=period).mean()
  rs = gain/loss
  return 100 - (100/(1+rs))

df['rsi'] = calculate_rsi(df['close'])
print(f"length of dataset before dropping:{len(df)}")
df = df.dropna()
print(f"length of dataset after dropping:{len(df)}")
print(df[['close','rsi']].head(10))

length of dataset before dropping:71897
length of dataset after dropping:71864
       close        rsi
13  13211.39  50.224748
14  13247.00  51.163172
15  13018.00  42.158961
16  13022.00  39.867769
17  13135.00  37.314969
18  13240.37  41.616146
19  13399.24  39.628874
20  13481.01  47.386741
21  13452.00  48.561459
22  13380.00  42.699059


#Volatility (Market Nervousness)


Volatility measures how much the price "swings" away from its average. In this project, we calculate the Rolling Standard Deviation of the returns over a 24-hour period.

The Logic for the Model:

    High Volatility: The price is jumping wildly (e.g., during a market crash or a massive pump). This indicates high uncertainty and "panic."

    Low Volatility: The price is moving in small, predictable steps. This indicates a "calm" or "boring" market.

Why the AI needs this:
Machine Learning models often behave differently in "quiet" markets vs. "chaotic" markets. By knowing the volatility, the AI can adjust its confidence. For example, a "Buy" signal during high volatility is much riskier than a "Buy" signal when things are calm.

In [16]:
# caculate 24 hours Rolling Volatility
df['volatility_24h'] = df['returns_scaled'].rolling(window=24).std()
print(f"length of dataset before dropping:{len(df)}")
#first 23 rows will be empty
df = df.dropna()
print(f"length of dataset after dropping:{len(df)}")

length of dataset before dropping:71864
length of dataset after dropping:71841


In [17]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 71841 entries, 36 to 71896
Data columns (total 16 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   timestamp         71841 non-null  object 
 1   close             71841 non-null  float64
 2   volume            71841 non-null  float64
 3   number_of_trades  71841 non-null  float64
 4   buy_pressure      71841 non-null  float64
 5   log_volume        71841 non-null  float64
 6   hour_sin          71841 non-null  float64
 7   hour_cos          71841 non-null  float64
 8   day_sin           71841 non-null  float64
 9   day_cos           71841 non-null  float64
 10  returns_scaled    71841 non-null  float64
 11  ema_12            71841 non-null  float64
 12  ema_26            71841 non-null  float64
 13  ema_gap           71841 non-null  float64
 14  rsi               71841 non-null  float64
 15  volatility_24h    71841 non-null  float64
dtypes: float64(15), object(1)
memory usage: 9.3+

##Price Gap (Recent Momentum)

The Price Gap (or Hourly Return) is the percentage change in the close price between the current hour and the previous hour.

In cryptocurrency, momentum is a powerful predictor. A sudden positive gap often signals the start of a "pump," while a negative gap can signal the beginning of a "sell-off." By providing this as an input feature, we allow the XGBoost model to weigh recent price velocity against our other indicators like RSI and EMA.

Note on Target vs. Feature:
While this column is mathematically similar to our target (returns_scaled), the Price Gap represents the past (what happened), whereas the target represents the future (what we want to predict).

In [18]:
#caculate price gap
df['price_gap']=df['close'].pct_change()

#drop first row that will be nan
df = df.dropna()

#display price gap ,returns scaled and close
df[['close','price_gap','returns_scaled']].tail()

,close,price_gap,returns_scaled
71892,71390.53,-0.005979,-1.217360
71893,71549.94,0.002233,0.438326
71894,71730.81,0.002528,0.497794
71895,72109.58,0.005280,1.052755
71896,72749.44,0.008873,1.777164


#Target - The "Future Shift"
In a real-world scenario, we use the information available now (at the close of an hour) to predict what will happen in the next hour.The Problem of Data Leakage:If we try to predict the returns_scaled of the current row, the AI will "cheat." It will see that the price just moved $+1\%$ and simply predict a positive return. This results in fake $100\%$ accuracy that fails the moment you try to trade live.The Solution: Shifting the Target:We shift the returns_scaled column upward by one row (shift(-1)).The Features ($X$) (RSI, EMA, Price Gap) now stay at Time $T$.The Target ($y$) now represents the return at Time $T+1$.By doing this, we force the XGBoost model to find patterns in current data that actually have "predictive power" for the future.

In [20]:
#define target
df['target'] = df['returns_scaled'].shift(-1)
#drop last row
df = df.dropna()
print(f"length of dataset {len(df)}")
#Define features (x) and target(y)
features = ['buy_pressure','log_volume','hour_sin','hour_cos',
            'day_sin','day_cos','ema_gap','rsi','volatility_24h','price_gap']

x = df[features]
y = df['target']
len(x),len(y)

length of dataset 71839


(71839, 71839)

In [21]:
x.head()

,buy_pressure,log_volume,hour_sin,hour_cos,day_sin,day_cos,ema_gap,rsi,volatility_24h,price_gap
37,0.569919,16.086897,-0.500000,-8.660254e-01,0.781831,0.62349,0.011396,47.535254,2.390473,-0.014467
38,0.599840,16.379462,-0.707107,-7.071068e-01,0.781831,0.62349,0.011225,52.791957,2.389970,0.001876
39,0.582625,16.457502,-0.866025,-5.000000e-01,0.781831,0.62349,0.003245,52.080988,2.284737,-0.007305
40,0.577226,16.231131,-0.965926,-2.588190e-01,0.781831,0.62349,0.014574,62.868340,2.337060,0.013979
41,0.559543,16.931902,-1.000000,-1.836970e-16,0.781831,0.62349,0.061477,82.577222,3.268459,0.058055


In [22]:
y.head()

,target
37,0.366298
38,-1.484744
39,2.806607
40,11.693021
41,2.100405
